In [1]:
import os
import h5py
import pandas as pd

BASE_DIR = './MillionSongSubset'


In [2]:
sve_pesme = []
brojac_gresaka = 0
print("kreni")


for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.endswith('.h5'):
            file_path = os.path.join(root, file)
       
            try:
                with h5py.File(file_path, 'r') as f:
                    meta_songs = f['metadata']['songs'][0]
                    raw_terms = f['metadata']['artist_terms'][:]
                    raw_artist_similarity = f['metadata']['similar_artists'][:]
                    raw_terms_weights = f['metadata']['artist_terms_weight'][:]
                    
                    song_id = meta_songs['song_id'].decode('utf-8', errors='replace')
                    artist_id =  meta_songs['artist_id'].decode('utf-8', errors='replace')
                    artist_terms = [term.decode('utf-8') for term in raw_terms]
                    artist_name = meta_songs['artist_name'].decode('utf-8', errors='replace')
                    artist_terms_weights = [weight for weight in raw_terms_weights]
                    analysis_songs = f['analysis']['songs'][0]
                    loudness = analysis_songs['loudness']
                    tempo = analysis_songs['tempo']
                    sim_art = [f.decode('utf-8') for f in raw_artist_similarity]
                    modes = int(analysis_songs['mode'])
                    mb_songs = f['musicbrainz']['songs'][0]
                    timesig = analysis_songs['time_signature']  
                    title = meta_songs['title'].decode('utf-8', errors='replace')
                    sve_pesme.append({

                       'song_id': song_id,
                        'artist_id': artist_id,
                        'title' : title,
                        'artist_name': artist_name,
                        'artist_terms': artist_terms,
                        'artist_terms_weight': artist_terms_weights,
                        'loudness': loudness,
                        'tempo': tempo,
                       'similar_artists': sim_art,
                        'mode': modes,
                        'time_signature': timesig
                    })
            except Exception as e:
                brojac_gresaka += 1
                if brojac_gresaka <= 3:
                    print(f"{file}: {e}")

print(f"grešaka: {brojac_gresaka}")

df = pd.DataFrame(sve_pesme)
print( {len(df)} )

if not df.empty:
    display(df.head())

kreni
grešaka: 0
{10000}


,song_id,artist_id,title,artist_name,artist_terms,artist_terms_weight,loudness,tempo,similar_artists,mode,time_signature
0,SOMZWCG12A8C13C480,ARD7TVE1187B99BFB1,I Didn't Mean To,Casual,"[hip hop, underground rap, g funk, alternative...","[1.0, 0.8979359555142553, 0.8842618474718359, ...",-11.197,92.198,"[ARV4KO21187FB38008, ARWHM281187FB3D381, ARJGO...",0,4
1,SOCIWDW12A8C13D406,ARMJAGH1187FB546F3,Soul Deep,The Box Tops,"[blue-eyed soul, pop rock, blues-rock, beach m...","[1.0, 0.8459884034332037, 0.8306895698215381, ...",-9.843,121.274,"[ARSZWK21187B9B26D7, ARLDW2Y1187B9B544F, ARG0T...",0,4
2,SOXVLOJ12AB0189215,ARKRRTF1187B9984DA,Amor De Cabaret,Sonora Santanera,"[salsa, cumbia, tejano, ranchera, latin pop, l...","[1.0, 0.9582578450180738, 0.9582578450180738, ...",-9.689,100.070,"[ARFSJUG11C8A421AAD, AR8SD041187FB36015, ARR75...",1,1
3,SONHOTT12A8C13493C,AR7G5I41187FB4CE6C,Something Girls,Adam Ant,"[pop rock, new wave, dance rock, rock, new rom...","[1.0, 0.9636972066614938, 0.9267729972686404, ...",-9.013,119.293,"[AR4R0741187FB39AF2, AR0D7K21187B9AD14E, ARRCB...",1,4
4,SOFSOCN12A8C143F5D,ARXR32B1187FB57099,Face the Ashes,Gob,"[pop punk, ska punk, breakcore, alternative me...","[1.0, 0.9609057433767874, 0.9592366232787087, ...",-4.501,129.738,"[ARUA62A1187B99D9B0, ARHJFFY1187B98BA76, ARHB1...",1,4


In [3]:
# 1. Prvo proveravamo ukupan broj praznih po kolonama
numericke_kolone = ['loudness', 'tempo', 'time_signature', 'mode']
print("--- BROJ PRAZNIH POLJA PO KOLONAMA ---")
print(df[numericke_kolone].isna().sum())
print("-" * 40)

# 2. Printamo tačne indekse redova gde se nalaze NaN vrednosti
print("\n--- LOKACIJE PRAZNIH POLJA (INDEKSI REDOVA) ---")
for kolona in numericke_kolone:
    prazni_redovi = df[df[kolona].isna()].index.tolist()
    
    if len(prazni_redovi) > 0:
        print(f"Kolona '{kolona}' ima prazna polja u sledećim redovima:")
        # Ako ima previše praznih, isprintaćemo samo prvih 20 da ne zatrpamo ekran
        if len(prazni_redovi) > 20:
            print(f"  {prazni_redovi[:20]} ... (i još {len(prazni_redovi) - 20} redova)")
        else:
            print(f"  {prazni_redovi}")
    else:
        print(f"Kolona '{kolona}' nema praznih polja.")

--- BROJ PRAZNIH POLJA PO KOLONAMA ---
loudness          0
tempo             0
time_signature    0
mode              0
dtype: int64
----------------------------------------

--- LOKACIJE PRAZNIH POLJA (INDEKSI REDOVA) ---
Kolona 'loudness' nema praznih polja.
Kolona 'tempo' nema praznih polja.
Kolona 'time_signature' nema praznih polja.
Kolona 'mode' nema praznih polja.


In [4]:
def uzmi_prvih_pet(terms):
    if isinstance(terms, list):
        return terms[:5]
    return []

df['artist_terms'] = df['artist_terms'].apply(uzmi_prvih_pet)

for i, red in df.head(3).iterrows():
    print(f"Pesma {red['song_id']}: {red['artist_terms']}")

df.head()



Pesma SOMZWCG12A8C13C480: ['hip hop', 'underground rap', 'g funk', 'alternative rap', 'gothic rock']
Pesma SOCIWDW12A8C13D406: ['blue-eyed soul', 'pop rock', 'blues-rock', 'beach music', 'soft rock']
Pesma SOXVLOJ12AB0189215: ['salsa', 'cumbia', 'tejano', 'ranchera', 'latin pop']


,song_id,artist_id,title,artist_name,artist_terms,artist_terms_weight,loudness,tempo,similar_artists,mode,time_signature
0,SOMZWCG12A8C13C480,ARD7TVE1187B99BFB1,I Didn't Mean To,Casual,"[hip hop, underground rap, g funk, alternative...","[1.0, 0.8979359555142553, 0.8842618474718359, ...",-11.197,92.198,"[ARV4KO21187FB38008, ARWHM281187FB3D381, ARJGO...",0,4
1,SOCIWDW12A8C13D406,ARMJAGH1187FB546F3,Soul Deep,The Box Tops,"[blue-eyed soul, pop rock, blues-rock, beach m...","[1.0, 0.8459884034332037, 0.8306895698215381, ...",-9.843,121.274,"[ARSZWK21187B9B26D7, ARLDW2Y1187B9B544F, ARG0T...",0,4
2,SOXVLOJ12AB0189215,ARKRRTF1187B9984DA,Amor De Cabaret,Sonora Santanera,"[salsa, cumbia, tejano, ranchera, latin pop]","[1.0, 0.9582578450180738, 0.9582578450180738, ...",-9.689,100.070,"[ARFSJUG11C8A421AAD, AR8SD041187FB36015, ARR75...",1,1
3,SONHOTT12A8C13493C,AR7G5I41187FB4CE6C,Something Girls,Adam Ant,"[pop rock, new wave, dance rock, rock, new rom...","[1.0, 0.9636972066614938, 0.9267729972686404, ...",-9.013,119.293,"[AR4R0741187FB39AF2, AR0D7K21187B9AD14E, ARRCB...",1,4
4,SOFSOCN12A8C143F5D,ARXR32B1187FB57099,Face the Ashes,Gob,"[pop punk, ska punk, breakcore, alternative me...","[1.0, 0.9609057433767874, 0.9592366232787087, ...",-4.501,129.738,"[ARUA62A1187B99D9B0, ARHJFFY1187B98BA76, ARHB1...",1,4


In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

if not df.empty:
    
    def pripremi_terms(terms_list):
        if not isinstance(terms_list, list) or len(terms_list) == 0: 
            return ""
        return " ".join([term.replace(" ", "_") for term in terms_list])

    df['terms_clean_text'] = df['artist_terms'].apply(pripremi_terms)

    tfidf = TfidfVectorizer(max_features=1000)
    tfidf_matrica = tfidf.fit_transform(df['terms_clean_text'])

    print("\n--- TF-IDF USPEŠNO ZAVRŠEN ---")
    print(f"Oblik TF-IDF matrice (Pesme x Karakteristike): {tfidf_matrica.shape}")

    nauceni_tagovi = tfidf.get_feature_names_out()
    print(f"Neki od naučenih tagova: {nauceni_tagovi[:10]}")


--- TF-IDF USPEŠNO ZAVRŠEN ---
Oblik TF-IDF matrice (Pesme x Karakteristike): (10000, 927)
Neki od naučenih tagova: ['00s' '2_tone' '60s' '60s_garage' '70s' '80s' '80s_pop' '90s' 'abstract'
 'accordion']


In [6]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[['loudness', 'tempo', 'mode', 'time_signature']] = scaler.fit_transform(df[['loudness', 'tempo', 'mode', 'time_signature']])

display(df.head())


,song_id,artist_id,title,artist_name,artist_terms,artist_terms_weight,loudness,tempo,similar_artists,mode,time_signature,terms_clean_text
0,SOMZWCG12A8C13C480,ARD7TVE1187B99BFB1,I Didn't Mean To,Casual,"[hip hop, underground rap, g funk, alternative...","[1.0, 0.8979359555142553, 0.8842618474718359, ...",0.774694,0.350792,"[ARV4KO21187FB38008, ARWHM281187FB3D381, ARJGO...",0.0,0.571429,hip_hop underground_rap g_funk alternative_rap...
1,SOCIWDW12A8C13D406,ARMJAGH1187FB546F3,Soul Deep,The Box Tops,"[blue-eyed soul, pop rock, blues-rock, beach m...","[1.0, 0.8459884034332037, 0.8306895698215381, ...",0.800628,0.461420,"[ARSZWK21187B9B26D7, ARLDW2Y1187B9B544F, ARG0T...",0.0,0.571429,blue-eyed_soul pop_rock blues-rock beach_music...
2,SOXVLOJ12AB0189215,ARKRRTF1187B9984DA,Amor De Cabaret,Sonora Santanera,"[salsa, cumbia, tejano, ranchera, latin pop]","[1.0, 0.9582578450180738, 0.9582578450180738, ...",0.803578,0.380743,"[ARFSJUG11C8A421AAD, AR8SD041187FB36015, ARR75...",1.0,0.142857,salsa cumbia tejano ranchera latin_pop
3,SONHOTT12A8C13493C,AR7G5I41187FB4CE6C,Something Girls,Adam Ant,"[pop rock, new wave, dance rock, rock, new rom...","[1.0, 0.9636972066614938, 0.9267729972686404, ...",0.816526,0.453882,"[AR4R0741187FB39AF2, AR0D7K21187B9AD14E, ARRCB...",1.0,0.571429,pop_rock new_wave dance_rock rock new_romantic
4,SOFSOCN12A8C143F5D,ARXR32B1187FB57099,Face the Ashes,Gob,"[pop punk, ska punk, breakcore, alternative me...","[1.0, 0.9609057433767874, 0.9592366232787087, ...",0.902948,0.493623,"[ARUA62A1187B99D9B0, ARHJFFY1187B98BA76, ARHB1...",1.0,0.571429,pop_punk ska_punk breakcore alternative_metal ...


In [7]:
import glob
import json
import pandas as pd

prvi_dataset_songs = set(
    df["artist_name"].astype(str).str.strip().str.lower()
    + " - "
    + df["title"].astype(str).str.strip().str.lower()
)

print(f"usepsno ucitane pesme {len(prvi_dataset_songs)}")


json_folder_path = r"C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\*.json"  # Primer: "C:/putanja/do/foldera/*.json"

json_songs = set()
json_files = glob.glob(json_folder_path)


if len(json_files) == 0:
    print("Python ne vidi ni jedan JSON fajl. Proveri da li je putanja 100% tačna!")
else:
    for file_path in json_files:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            if "playlists" not in data:
                print(f"Fajl {file_path} nema ključ 'playlists'. Ključevi su: {list(data.keys())}")
                continue
                
            playlists_list = data.get("playlists", [])
            print(f"fajl {file_path} br plejlisti: {len(playlists_list)}")
            
            for playlist in playlists_list:
                tracks_list = playlist.get("tracks", [])
                for track in tracks_list:
                    artist = track.get("artist_name")
                    track_name = track.get("track_name")
                    
                    if artist is None or track_name is None:
                        print(f"dostupni ključevi: {list(track.keys())}")
                        break
                        
                    song_key = str(artist).strip().lower() + " - " + str(track_name).strip().lower()
                    json_songs.add(song_key)

print(f"\nUkupan broj unikatnih pesama uspešno pokupljenih iz JSON-a: {len(json_songs)}")

usepsno ucitane pesme 9939
fajl C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\mpd.slice.0-999.json br plejlisti: 89
fajl C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\mpd.slice.1000-1999.json br plejlisti: 80
fajl C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\mpd.slice.10000-10999.json br plejlisti: 72
fajl C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\mpd.slice.100000-100999.json br plejlisti: 96
fajl C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\mpd.slice.101000-101999.json br plejlisti: 82
fajl C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\mpd.slice.102000-102999.json br plejlisti: 97
fajl C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\mpd.slice.103000-103999.json br plejlisti: 84
fajl C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlis

In [8]:
import pandas as pd
k = pd.read_csv(r'C:\Users\Lena\Downloads\millionsongsubset\train_triplets.txt\train_triplets.txt', sep='\t', names=['user_id', 'song_id', 'play_count'], nrows=1000000)
print(k['song_id'].nunique())
pesme_u_datasetu_1 = set(df['song_id'])

print(k['song_id'].nunique())

k = k[k['song_id'].isin(pesme_u_datasetu_1)].copy()

print(k['song_id'].nunique())
k = k.reset_index(drop=True)

if not k.empty:
    display(k.head())
    print(k['song_id'].nunique())

148039
148039
1580


,user_id,song_id,play_count
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOWEZSI12A81C21CE6,1
1,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SODCXXY12AB0187452,2
2,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SOWPAXV12A67ADA046,18
3,b64cdd1a0bd907e5e00b39e345194768e330d652,SOLXDDC12A6701FBFD,1
4,b64cdd1a0bd907e5e00b39e345194768e330d652,SONJBQX12A6D4F8382,4


1580


In [9]:

zajednicki_kljucevi = prvi_dataset_songs.keys() & json_songs.keys() if isinstance(prvi_dataset_songs, dict) else set(prvi_dataset_songs).intersection(set(json_songs))


lista_zajednickih = []
for kljuc in zajednicki_kljucevi:
    if isinstance(prvi_dataset_songs, dict) and isinstance(json_songs, dict):
        lista_zajednickih.append({
            'artist_name': prvi_dataset_songs[kljuc]['artist'],
            'song_name': prvi_dataset_songs[kljuc]['title']
        })
    else:
        delovi = kljuc.split(" - ")
        if len(delovi) >= 2:
            lista_zajednickih.append({
                'artist_name': delovi[0].title(),
                'song_name': " - ".join(delovi[1:]).title()
            })

df_zajednicke = pd.DataFrame(lista_zajednickih)
df_zajednicke.head(400000)
df_zajednicke.to_csv(
    r"C:\Users\Lena\Downloads\millionsongsubset\zajednicke_pesme.csv",
    index=False,
    encoding="utf-8-sig",
)
print(len(df_zajednicke))

1767


In [10]:

putanja_csv = r"C:\Users\Lena\Downloads\millionsongsubset\zajednicke_pesme.csv"
df_zajednicke = pd.read_csv(putanja_csv)

dozvoljene_pesme = set(
    df_zajednicke["artist_name"].astype(str).str.strip().str.lower()
    + " - "
    + df_zajednicke["song_name"].astype(str).str.strip().str.lower()
)

print(f"uvitano {len(dozvoljene_pesme)} zaj pesama")

json_folder_path = r"C:\Users\Lena\Downloads\millionsongsubset\spotify_million_playlist_dataset\data\*.json"
json_files = glob.glob(json_folder_path)

print(f"nađeno {len(json_files)} JSON fajlova ")

ukupno_pre = 0
ukupno_posle = 0

for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    fajl_izmenjen = False

    
    for playlist in data.get("playlists", []):
        ociscene_trake = []
        tracks_list = playlist.get("tracks", [])
        ukupno_pre += len(tracks_list)

        for track in tracks_list:
            art_json = str(track.get("artist_name", "")).strip().lower()
            track_json = str(track.get("track_name", "")).strip().lower()
            kljuc = f"{art_json} - {track_json}"

            
            if kljuc in dozvoljene_pesme:
                ociscene_trake.append(track)
            else:
                fajl_izmenjen = True

        playlist["tracks"] = ociscene_trake
        ukupno_posle += len(ociscene_trake)

    if fajl_izmenjen:
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

print("=" * 60)
print(f"br pesama pre brisanja: {ukupno_pre}")
print(f"br pesama posle brisanja: {ukupno_posle}")
print(f"izbrisano: {ukupno_pre - ukupno_posle}")
print("=" * 60)

uvitano 1767 zaj pesama
nađeno 1000 JSON fajlova 
br pesama pre brisanja: 256136
br pesama posle brisanja: 256136
izbrisano: 0


#### SPAJAMO TRAIN_TRIPLETS.TXT I JSON FAJLOVE

In [11]:
df_interactions = pd.read_csv(r'C:\Users\Lena\Downloads\millionsongsubset\train_triplets.txt\train_triplets.txt', sep='\t', names=['user_id', 'song_id', 'play_count'], nrows=1000000)

pesme_u_datasetu_1 = set(df['song_id'])


print(df_interactions['song_id'].nunique())

df_interactions = df_interactions[df_interactions['song_id'].isin(pesme_u_datasetu_1)].copy()

print(df_interactions['song_id'].nunique())
df_filtriran = df_interactions.reset_index(drop=True)

if not df_filtriran.empty:
    display(df_filtriran.head(150000))


# #dodajemo podatke i z json fajlova u train_triplets
# for row in df_interactions.itertuples():
#     user_id = row.user_id
#     song_id = row.song_id
#     play_count = 1

# df_interactions.head(1000000000)

148039
1580


,user_id,song_id,play_count
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOWEZSI12A81C21CE6,1
1,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SODCXXY12AB0187452,2
2,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SOWPAXV12A67ADA046,18
3,b64cdd1a0bd907e5e00b39e345194768e330d652,SOLXDDC12A6701FBFD,1
4,b64cdd1a0bd907e5e00b39e345194768e330d652,SONJBQX12A6D4F8382,4
...,...,...,...
15690,bacbde7b7454169ea272cfdf38c422132ce9fd56,SONQBUB12A6D4F8ED0,1
15691,bacbde7b7454169ea272cfdf38c422132ce9fd56,SOULTKQ12AB018A183,3
15692,ed9b75f9343cea8debc3db5ca85f846a373252a2,SOLXDDC12A6701FBFD,1
15693,e447f9617f74ce8b363e27f229aee04f39fec88c,SOOPVJI12AB0183957,1


In [12]:
import glob
import json
import pandas as pd

rows = []


for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
        playlists_list = data.get("playlists", [])
        for playlist in playlists_list:
            user_id = playlist.get("name")
            
            tracks_list = playlist.get("tracks", [])
            for track in tracks_list:
                artist = track.get("artist_name")
                track_name = track.get("track_name")
                
                if artist and track_name:
                    song_id = f"{artist.strip()} - {track_name.strip()}"
                    
                    rows.append({
                        "user_id": user_id,
                        "song_id": song_id,
                        "artist_name": artist.strip(),
                        "play_count": 1
                    })

df_json = pd.DataFrame(rows)
df_proba = pd.concat([df_filtriran, df_json], ignore_index=True)
print(f"redova {len(df_filtriran)}")

print(df_proba.head(1000000))
print (df_proba['song_id'].nunique())

redova 15695
                                         user_id  \
0       b80344d063b5ccb3212f76538f3d9e43d87dca9e   
1       4bd88bfb25263a75bbdd467e74018f4ae570e5df   
2       4bd88bfb25263a75bbdd467e74018f4ae570e5df   
3       b64cdd1a0bd907e5e00b39e345194768e330d652   
4       b64cdd1a0bd907e5e00b39e345194768e330d652   
...                                          ...   
271826                                       80s   
271827                                      Lake   
271828                                      Lake   
271829                                      Lake   
271830                                      Lake   

                                                  song_id  play_count  \
0                                      SOWEZSI12A81C21CE6           1   
1                                      SODCXXY12AB0187452           2   
2                                      SOWPAXV12A67ADA046          18   
3                                      SOLXDDC12A6701FBFD           1 

In [13]:
df['full_name_clean'] = (df['artist_name'].astype(str).str.strip() + " - " + df['title'].astype(str).str.strip()).str.lower()
losi = df_proba.copy()

mapping = df.drop_duplicates(subset=['full_name_clean']).set_index('full_name_clean')['song_id']

losi['song_id_clean'] = losi['song_id'].astype(str).str.strip().str.lower()
losi['song_id'] = losi['song_id_clean'].map(mapping)
df.drop(columns=['full_name_clean'], inplace=True)
losi.drop(columns=['song_id_clean'], inplace=True)

losi = losi.dropna(subset=['song_id'])

print(losi['song_id'].nunique())
losi.head(100000)

1767


,user_id,song_id,play_count,artist_name
15695,Wedding,SOSGQPS12A8AE48538,1,Chris Brown
15696,Wedding,SOFQBMN12A8C1428AD,1,Beyoncé
15697,Wedding,SOLFDCM12A6D4F7A7C,1,Tom Petty and the Heartbreakers
15698,90's,SOLIQUE12A58A78306,1,Incubus
15699,90's,SOSMWDQ12A6701E981,1,Nirvana
...,...,...,...,...
115690,Playlist #1,SOQKBQI12AB0182711,1,Britney Spears
115691,Playlist #1,SOUEHCD12AB0188F80,1,3 Doors Down
115692,Playlist #1,SOOTADG12AF729F5CF,1,Céline Dion
115693,pls,SOFQRSV12A8C1358A9,1,Fall Out Boy


In [14]:
import pandas as pd
# 1. Iz df_proba izbacujemo sve redove koji su bili "pokvareni" (oni koji imaju ' - ' u song_id)
# Sa ~ obrnemo uslov, pa zadržavamo samo one koji su od starta bili dobri
df_samo_dobri = df_proba[~df_proba['song_id'].astype(str).str.contains(' - ', na=False)]
df_filtriran = pd.concat([df_samo_dobri, losi], ignore_index = True)
# 2. Sada spajamo te dobre redove sa tvojim ispravljenim df_filtriran datasetom
print(df_samo_dobri['song_id'].nunique())

# df_filtriran sada ima sve pesme ispravljene, bez dupliranja i bez starih pokvarenih stringova!
# df_filtriran.head(1000000)
print(df_filtriran['song_id'].nunique())

1580
2700


In [15]:
csv_output_path = "filtriran_dataset.csv"
df_filtriran.to_csv(csv_output_path, index=False, encoding="utf-8")


#### RADIMO COUSIN SIM ZA SPOJENE DATASETOVE

In [16]:
import pandas as pd

df_karakteristike = df[
    ["song_id", "artist_id", "artist_terms", "loudness","mode", "tempo", "time_signature", "similar_artists"]
].copy()

df_karakteristike = df_karakteristike.drop_duplicates(
    subset=["song_id"]
)

df_finalni = pd.merge(
    df_filtriran,
    df_karakteristike,
    left_on=["song_id"],
    right_on=["song_id"],
    how="left",
)

if "title" in df_finalni.columns:
    df_finalni = df_finalni.drop(columns=["title"])


display(df_finalni.head(1570000))
csv_output_path = "mergeovan.csv"
df_finalni.to_csv(csv_output_path, index=False, encoding="utf-8")

,user_id,song_id,play_count,artist_name,artist_id,artist_terms,loudness,mode,tempo,time_signature,similar_artists
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOWEZSI12A81C21CE6,1,NaN,AR2UQQ51187B9AC816,"[flamenco, soundtrack, folk, spanish, acoustic]",0.828210,0.0,0.627810,0.142857,"[AR9Z7JB1187B99DB3D, ARC1SF21187FB51D0F, AR0F5..."
1,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SODCXXY12AB0187452,2,NaN,ARXLMH011C8A415658,"[pop rap, crunk, rapcore, screamo, breakcore]",0.767205,1.0,0.455096,0.571429,"[ARHAUVU122BCFCBA38, AR258TI11C8A416B5B, ARX94..."
2,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SOWPAXV12A67ADA046,18,NaN,ARUQ6301187FB54EBA,"[pop rap, hip hop, hip house, new jack swing, ...",0.880595,1.0,0.485477,0.571429,"[ARNHMFD1187FB3B3F6, ARELPXQ1187FB384FD, ARX9Y..."
3,b64cdd1a0bd907e5e00b39e345194768e330d652,SOLXDDC12A6701FBFD,1,NaN,ARTH9041187FB43E1F,"[hip hop, rap, hardcore rap, club, soundtrack]",0.912755,0.0,0.685498,0.571429,"[AR23C041187FB4D534, ARZER7I1187FB385AF, ARDD0..."
4,b64cdd1a0bd907e5e00b39e345194768e330d652,SONJBQX12A6D4F8382,4,NaN,ARF8HTQ1187B9AE693,"[techno, electronica, electronic, pop, french]",0.893026,0.0,0.423094,0.571429,"[AR3NPVS1187FB5108F, ARMKBL21187FB38230, ARYJ8..."
...,...,...,...,...,...,...,...,...,...,...,...
271826,80s,SOCWJDB12A58A776AF,1,Rick Astley,ARWPYQI1187FB4D55A,"[dance pop, rock, pop, england, adult contempo...",0.840717,1.0,0.431305,0.571429,"[ARCJMMQ1187B98D1FD, ARY7CAU1187B98A4F6, ARIJP..."
271827,Lake,SOLFDCM12A6D4F7A7C,1,Tom Petty and the Heartbreakers,ARBEBBY1187B9B43DB,"[heartland rock, jam band, pop rock, southern ...",0.817503,0.0,0.642801,0.571429,"[ARSAIUN1187FB398D3, ARF4L041187FB4D318, AR91C..."
271828,Lake,SOOTBSZ12A8C140223,1,Daryl Hall & John Oates,ARN0DMU1187FB5B63A,"[pop rock, blue-eyed soul, soft rock, folk roc...",0.879120,1.0,0.680388,0.571429,"[AR4U91L1187B9A2401, ARNCXKC1187FB38C42, ARUIH..."
271829,Lake,SOEYHBE12AB01848F6,1,Maroon 5,ARF5M7Q1187FB501E8,"[pop, rock, alternative, modern rock, club]",0.873470,0.0,0.437541,0.571429,"[ARY53RR1187B9AE485, ARVLXWP1187FB5B94A, ARWK5..."


In [17]:
# Broj redova koji su identični nekim prethodnim redovima
broj_duplikata = df_filtriran.duplicated().sum()

print(f"Ukupno ima {broj_duplikata} identičnih redova.")
# Briše identične redove i čuva samo prvu pojavu svakog reda
pr_cist = df_filtriran.drop_duplicates()

# Ako želiš da izmene ostanu direktno u originalnom pr datasetu, dodaj inplace=True:
# pr.drop_duplicates(inplace=True)

print(f"Broj redova pre čišćenja: {len(df_filtriran)}")
df_filtriran = pr_cist  # Ažuriramo df_filtriran sa očišćenim podacima
print(f"Broj redova nakon čišćenja: {len(df_filtriran)}")


Ukupno ima 120362 identičnih redova.
Broj redova pre čišćenja: 271831
Broj redova nakon čišćenja: 151469


In [18]:
csv_output_path = "filtriran_dataset.csv"
df_filtriran.to_csv(csv_output_path, index=False, encoding="utf-8")

In [19]:
numericke_kolone = ['loudness', 'tempo', 'mode','time_signature']
print("--- BROJ PRAZNIH POLJA PO KOLONAMA ---")
print(df_finalni[numericke_kolone].isna().sum())
print("-" * 40)

# 2. Printamo tačne indekse redova gde se nalaze NaN vrednosti
print("\n--- LOKACIJE PRAZNIH POLJA (INDEKSI REDOVA) ---")
for kolona in numericke_kolone:
    prazni_redovi = df_finalni[df_finalni[kolona].isna()].index.tolist()
    
    if len(prazni_redovi) > 0:
        print(f"Kolona '{kolona}' ima prazna polja u sledećim redovima:")
        # Ako ima previše praznih, isprintaćemo samo prvih 20 da ne zatrpamo ekran
        if len(prazni_redovi) > 20:
            print(f"  {prazni_redovi[:20]} ... (i još {len(prazni_redovi) - 20} redova)")
        else:
            print(f"  {prazni_redovi}")
    else:
        print(f"Kolona '{kolona}' nema praznih polja.")

--- BROJ PRAZNIH POLJA PO KOLONAMA ---
loudness          0
tempo             0
mode              0
time_signature    0
dtype: int64
----------------------------------------

--- LOKACIJE PRAZNIH POLJA (INDEKSI REDOVA) ---
Kolona 'loudness' nema praznih polja.
Kolona 'tempo' nema praznih polja.
Kolona 'mode' nema praznih polja.
Kolona 'time_signature' nema praznih polja.


In [20]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

if not df_finalni.empty:

    def pripremi_terms(terms_list):
        if not isinstance(terms_list, list) or len(terms_list) == 0:
            return ""
        return " ".join([term.replace(" ", "_") for term in terms_list])

    df_finalni["terms_clean_text"] = df_finalni["artist_terms"].apply(
        pripremi_terms
    )

    tfidf = TfidfVectorizer(max_features=1000)
    tfidf_matrica = tfidf.fit_transform(df_finalni["terms_clean_text"])

    ts_min = df_finalni["time_signature"].fillna(0).min()
    ts_max = df_finalni["time_signature"].fillna(0).max()

    if ts_max != ts_min:
        time_sig_norm = (df_finalni["time_signature"].fillna(0) - ts_min) / (
            ts_max - ts_min
        )
    else:
        time_sig_norm = df_finalni["time_signature"].fillna(0)

    l_norm = df_finalni["loudness"].values
    t_norm = df_finalni["tempo"].values
    m_norm = df_finalni["mode"].values
    ts_norm = time_sig_norm.values

    numericke_karakteristike = np.vstack([l_norm, t_norm, m_norm,ts_norm]).T
    numericke_sparse = csr_matrix(numericke_karakteristike)

    tfidf_matrix = hstack([tfidf_matrica, numericke_sparse]).tocsr()

    print(
        f"Pesme x Sve karakteristike {tfidf_matrix.shape}"
    )

Pesme x Sve karakteristike (271831, 636)


In [21]:
def generisi_content_based_preporuke_safe(
    target_user_id,
    interakcije_df,
    konacni_vektori_pesama,
    song_id_u_indeks,
    df_pesme,
    top_n=10,
):
    korisnik_istorija = interakcije_df[
        interakcije_df["user_id"] == target_user_id
    ]

    if korisnik_istorija.empty:
        print(f"user {target_user_id} nije muzikalan.")
        return []

    profil_korisnika = None
    ukupno_pustanja = 0

    for _, red in korisnik_istorija.iterrows():
        s_id = red["song_id"]
        #broj_slusanja = red["play_count"]

        if s_id in song_id_u_indeks:
            indeks_pesme = song_id_u_indeks[s_id]
            vektor_pesme = konacni_vektori_pesama.getrow(indeks_pesme)

            if profil_korisnika is None:
                profil_korisnika = vektor_pesme #* broj_slusanja
            else:
                profil_korisnika += vektor_pesme #* broj_slusanja

            #ukupno_pustanja += broj_slusanja

    if profil_korisnika is None:
        print(f"user {target_user_id} nije muzikalan.")
        return []

   # profil_korisnika = profil_korisnika / ukupno_pustanja


    konacni_vektori_pesama.data = np.nan_to_num(konacni_vektori_pesama.data)
    profil_korisnika.data = np.nan_to_num(profil_korisnika.data)

    skorovi_slicnosti = cosine_similarity(
        profil_korisnika, konacni_vektori_pesama
    )[0]
    rangirani_indeksi = np.argsort(skorovi_slicnosti)[::-1]

    
    slusane_pesme_ids = set(korisnik_istorija["song_id"])
    vec_preporuceno_ids = set()

    preporuke = []

    kolona_naslov = (
        "song_name"
        if "song_name" in df_pesme.columns
        else (
            "title"
            if "title" in df_pesme.columns
            else (
                "song_title" if "song_title" in df_pesme.columns else None
            )
        )
    )

    for indeks in rangirani_indeksi:
        pesma_info = df_pesme.iloc[indeks]
        s_id = pesma_info["song_id"]

        if s_id not in slusane_pesme_ids and s_id not in vec_preporuceno_ids:

            prikazni_naslov = (
                pesma_info[kolona_naslov]
                if kolona_naslov
                else s_id
            )

            preporuke.append(
                {
                    "title": prikazni_naslov,
                    "song_id": s_id,
                    "score": skorovi_slicnosti[indeks],
                }
            )

            vec_preporuceno_ids.add(s_id)

        if len(preporuke) == top_n:
            break

    return preporuke

In [22]:
tekstualni_nizovi = df_finalni["artist_terms"].apply(
    lambda x: (
        " ".join([tag.replace(" ", "_") for tag in x])
        if isinstance(x, list)
        else ""
    )
)

song_id_u_indeks = {
    song_id: idx for idx, song_id in enumerate(df_finalni["song_id"])
}


pesme_u_matrici = set(df_finalni["song_id"].unique())
df_interactions_sigurno = df_finalni[
    df_finalni["song_id"].isin(pesme_u_matrici)
].copy()

target_user_id = df_interactions_sigurno["user_id"].value_counts().index[0]
print(f"Računam nove preporuke za korisnika: {target_user_id}...")


preporuke_napredne = generisi_content_based_preporuke_safe(
    target_user_id=target_user_id,
    interakcije_df=df_interactions_sigurno,
    konacni_vektori_pesama=tfidf_matrix,  
    song_id_u_indeks=song_id_u_indeks,
    df_pesme=df_finalni,
    top_n=10,
)


if preporuke_napredne:
    print(
        f"\ntop 10"
    )
    print("-" * 75)
    for rank, p in enumerate(preporuke_napredne, 1):
        print(
            f"{rank}. {p['title']} [{p['song_id']}] (Sličnost: {p['score']:.4f})"
        )
else:
    print("\n uh ne treba ovo ovako")

Računam nove preporuke za korisnika: Rock...

top 10
---------------------------------------------------------------------------
1. SOQKQJU12A58A7D262 [SOQKQJU12A58A7D262] (Sličnost: 0.8820)
2. SOUVHAM12A6D4FA42A [SOUVHAM12A6D4FA42A] (Sličnost: 0.8780)
3. SOJOBVT12A6701E062 [SOJOBVT12A6701E062] (Sličnost: 0.8741)
4. SORIYMI12AB0185013 [SORIYMI12AB0185013] (Sličnost: 0.8734)
5. SOHWVJJ12AB0185F6D [SOHWVJJ12AB0185F6D] (Sličnost: 0.8689)
6. SOOCZWD12AB0185147 [SOOCZWD12AB0185147] (Sličnost: 0.8679)
7. SOXLEKB12A8C1338EE [SOXLEKB12A8C1338EE] (Sličnost: 0.8669)
8. SOFOAIA12A8C136186 [SOFOAIA12A8C136186] (Sličnost: 0.8653)
9. SOWWLVP12A8C138BFB [SOWWLVP12A8C138BFB] (Sličnost: 0.8630)
10. SOTOMJM12A8C134AA2 [SOTOMJM12A8C134AA2] (Sličnost: 0.8620)


In [23]:
k =  pd.read_csv(r'C:\Users\Lena\Downloads\millionsongsubset\mergeovan.csv')
k.head()

,user_id,song_id,play_count,artist_name,artist_id,artist_terms,loudness,mode,tempo,time_signature,similar_artists
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOWEZSI12A81C21CE6,1,NaN,AR2UQQ51187B9AC816,"['flamenco', 'soundtrack', 'folk', 'spanish', ...",0.828210,0.0,0.627810,0.142857,"['AR9Z7JB1187B99DB3D', 'ARC1SF21187FB51D0F', '..."
1,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SODCXXY12AB0187452,2,NaN,ARXLMH011C8A415658,"['pop rap', 'crunk', 'rapcore', 'screamo', 'br...",0.767205,1.0,0.455096,0.571429,"['ARHAUVU122BCFCBA38', 'AR258TI11C8A416B5B', '..."
2,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SOWPAXV12A67ADA046,18,NaN,ARUQ6301187FB54EBA,"['pop rap', 'hip hop', 'hip house', 'new jack ...",0.880595,1.0,0.485477,0.571429,"['ARNHMFD1187FB3B3F6', 'ARELPXQ1187FB384FD', '..."
3,b64cdd1a0bd907e5e00b39e345194768e330d652,SOLXDDC12A6701FBFD,1,NaN,ARTH9041187FB43E1F,"['hip hop', 'rap', 'hardcore rap', 'club', 'so...",0.912755,0.0,0.685498,0.571429,"['AR23C041187FB4D534', 'ARZER7I1187FB385AF', '..."
4,b64cdd1a0bd907e5e00b39e345194768e330d652,SONJBQX12A6D4F8382,4,NaN,ARF8HTQ1187B9AE693,"['techno', 'electronica', 'electronic', 'pop',...",0.893026,0.0,0.423094,0.571429,"['AR3NPVS1187FB5108F', 'ARMKBL21187FB38230', '..."


In [24]:
def kreiraj_train_test_interakcije(df_interakcije, test_pesama_po_korisniku=2):
    train_list = []
    test_list = []
    
    for user_id, group in df_interakcije.groupby('user_id'):
        if len(group) <= test_pesama_po_korisniku:
            train_list.append(group)
        else:
            test_uzorak = group.sample(n=test_pesama_po_korisniku, random_state=42)
            train_uzorak = group.drop(test_uzorak.index)
            
            train_list.append(train_uzorak)
            test_list.append(test_uzorak)
            
    df_train = pd.concat(train_list).reset_index(drop=True)
    df_test = pd.concat(test_list).reset_index(drop=True)
    
    return df_train, df_test


df_train, df_test = kreiraj_train_test_interakcije(df_interactions_sigurno, test_pesama_po_korisniku=2)

print(f"tnterakcije treninga {len(df_train)}")
print(f"interakcije testa{len(df_test)}")

tnterakcije treninga 240127
interakcije testa31704


In [25]:
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

def evaluiraj_maksimalni_cb(
    df_train,
    df_test,
    kombinovana_matrica,
    song_id_u_indeks,
    df_pesme,
    top_n=10,
):
    ukupno_hitova = 0
    ukupno_preciznost = 0.0
    procenjeni_korisnici = 0

    test_users = df_test["user_id"].unique()

    # kolona_slusanja = (
    #     "play_count" if "play_count" in df_train.columns else "listen_count"
    # )
    kombinovana_matrica = kombinovana_matrica.tocsr()

    kombinovana_matrica.data = np.nan_to_num(kombinovana_matrica.data)

    for target_user_id in test_users:
        user_train = df_train[df_train["user_id"] == target_user_id]
        if user_train.empty:
            continue

        user_test_pesme = set(
            df_test[df_test["user_id"] == target_user_id]["song_id"]
        )

        profil_korisnika = sp.csr_matrix(
            (1, kombinovana_matrica.shape[1]), dtype=np.float64
        )
        ukupno_pustanja = 0

        for _, red in user_train.iterrows():
            s_id = red["song_id"]
            # broj_slusanja = red[kolona_slusanja]

            if s_id in song_id_u_indeks:
                idx_p = song_id_u_indeks[s_id]
                profil_korisnika += (
                    kombinovana_matrica.getrow(idx_p) #* broj_slusanja
                )
                #ukupno_pustanja += broj_slusanja

        #if ukupno_pustanja == 0:
        #    continue

        #profil_korisnika = profil_korisnika / ukupno_pustanja

        skorovi = cosine_similarity(
            profil_korisnika, kombinovana_matrica
        ).flatten()
        top_indeksi = np.argsort(skorovi)[::-1]

        skup_slusanih_id = set(user_train["song_id"])
        preporuke_id = []
        vec_dodato = set()

        for idx in top_indeksi:
            pesma_info = df_pesme.iloc[idx]
            s_id = pesma_info["song_id"]

            if s_id not in skup_slusanih_id and s_id not in vec_dodato:
                preporuke_id.append(s_id)
                vec_dodato.add(s_id)

            if len(preporuke_id) == top_n:
                break

        pogoci = [pesma for pesma in preporuke_id if pesma in user_test_pesme]
        broj_pogodaka = len(pogoci)

        if broj_pogodaka > 0:
            ukupno_hitova += 1
        ukupno_preciznost += broj_pogodaka / top_n
        procenjeni_korisnici += 1

    prosecan_hr = (
        ukupno_hitova / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    )
    prosecna_preciznost = (
        ukupno_preciznost / procenjeni_korisnici
        if procenjeni_korisnici > 0
        else 0
    )
    
    
    print(f"Evaluirano korisnika: {procenjeni_korisnici}")
    print(f"Hit Rate@{top_n}:     {prosecan_hr:.4f}")
    print(f"Precision@{top_n}:    {prosecna_preciznost:.4f}")

    return prosecan_hr, prosecna_preciznost


izvor_interakcija = df_finalni


train_list = []
#val_list = []
test_list = []

for user_id, group in izvor_interakcija.groupby("user_id"):
    n_samples = len(group)

    if n_samples >= 6:
        # u_train, u_ostatak = train_test_split(
        #     group, test_size=0.30, shuffle=True, random_state=42
        # )
        # u_val, u_test = train_test_split(
        #     u_ostatak, test_size=0.50, shuffle=True, random_state=42
        # )
        u_train, u_test = train_test_split(
           group, test_size=0.30, shuffle=True, random_state=42
        )
        train_list.append(u_train)
        # val_list.append(u_val)
        test_list.append(u_test)

    # 2. Korisnici sa 3, 4 ili 5 pesama - ručno raspoređujemo da izbegnemo prazne setove
    elif 3 <= n_samples <= 5:
        # Promešamo grupu pre ručne podele
        group_shuffled = group.sample(frac=1, random_state=42)

        # Ovdje uvek dajemo 1 pesmu u val, 1 u test, a ostatak (1, 2 ili 3 pesme) ide u train
        u_val = group_shuffled.iloc[[0]]
        u_test = group_shuffled.iloc[[1]]
        u_train = group_shuffled.iloc[2:]

        train_list.append(u_train)
        # val_list.append(u_val)
        test_list.append(u_test)

    # 3. Korisnici sa tačno 2 pesme - 1 u train, 1 u test (val ostaje prazan za njih)
    elif n_samples == 2:
        u_train, u_test = train_test_split(
            group, test_size=0.50, shuffle=True, random_state=42
        )
        train_list.append(u_train)
        test_list.append(u_test)

    
    else:
        train_list.append(group)


df_train = pd.concat(train_list, ignore_index=True)
# df_val = pd.concat(val_list, ignore_index=True) if val_list else pd.DataFrame()
df_test = pd.concat(test_list, ignore_index=True)

print(f"Veličina train seta:       {df_train.shape[0]} interakcija")
# print(f"Veličina validation seta:  {df_val.shape[0]} interakcija")
print(f"Veličina test seta:        {df_test.shape[0]} interakcija\n")
 

hr_full, prec_full = evaluiraj_maksimalni_cb(
    df_train=df_train,
    df_test=df_test,  
    kombinovana_matrica=tfidf_matrix,
    song_id_u_indeks=song_id_u_indeks,
    df_pesme=df_finalni,
    top_n=15,
)

Veličina train seta:       177091 interakcija
Veličina test seta:        86283 interakcija



KeyboardInterrupt: 

In [ ]:
from sklearn.metrics.pairwise import manhattan_distances
import numpy as np
import pandas as pd

def generisi_manhattan_preporuke_safe(target_user_id, interakcije_df, konacni_vektori_pesama, song_id_u_indeks, df_pesme, top_n=10):
    # 1. Pronalazimo sve pesme koje je ovaj korisnik slušao
    korisnik_istorija = interakcije_df[interakcije_df['user_id'] == target_user_id]
    
    if korisnik_istorija.empty:
        return []

    # 2. Pravimo profil korisnika kao ponderisani prosek vektora pesama
    profil_korisnika = None
    ukupno_pustanja = 0
    
    for _, red in korisnik_istorija.iterrows():
        s_id = red['song_id']
        broj_slusanja = red['play_count']
        
        # Proveravamo da li pesma postoji u našoj matrici karakteristika
        if s_id in song_id_u_indeks:
            indeks_pesme = song_id_u_indeks[s_id]
            
            # Izvlačimo vektor te pesme iz CSR sparse matrice
            vektor_pesme = konacni_vektori_pesama.getrow(indeks_pesme).toarray()
            
            # Ponderišemo vektor brojem slušanja
            if profil_korisnika is None:
                profil_korisnika = vektor_pesme * broj_slusanja
            else:
                profil_korisnika += vektor_pesme * broj_slusanja
                
            ukupno_pustanja += broj_slusanja

    if profil_korisnika is None or ukupno_pustanja == 0:
        return []
        
    # Normalizacija profila korisnika
    profil_korisnika = profil_korisnika / ukupno_pustanja

    # 3. Računamo Menhetn rastojanje između profila i SVIH pesama u bazi
    # .toarray() se koristi jer manhattan_distances zahteva guste nizove
    udaljenosti = manhattan_distances(profil_korisnika, konacni_vektori_pesama.toarray())[0]

    # 4. Sortiramo pesme: manja udaljenost znači VEĆU sličnost!
    # argsort prirodno sortira od najmanjeg ka najvećem broju
    rangirani_indeksi = np.argsort(udaljenosti)

    # 5. Izvlačimo top N pesama, preskačući one koje je već slušao
    slusane_pesme_ids = set(korisnik_istorija['song_id'])
    preporuke = []
    
    for indeks in rangirani_indeksi:
        pesma_info = df_pesme.iloc[indeks]
        if pesma_info['song_id'] not in slusane_pesme_ids:
            preporuke.append({
                'song_id': pesma_info['song_id'],
                'distance': udaljenosti[indeks]  # Beležimo udaljenost (skor)
            })
        
        if len(preporuke) == top_n:
            break
            
    return preporuke

In [ ]:
# Pronalazimo najaktivnijeg korisnika (isto kao u tvom kodu)
target_user_id = df_interactions_sigurno['user_id'].value_counts().index[0]

print(f"Računam nove Menhetn preporuke za korisnika: {target_user_id}...")

# Pozivamo novu Menhetn funkciju
preporuke_menhetn = generisi_manhattan_preporuke_safe(
    target_user_id=target_user_id,
    interakcije_df=df_interactions_sigurno,
    konacni_vektori_pesama=kombinovana_matrica,  # Tvoja spojena matrica
    song_id_u_indeks=song_id_u_indeks,
    df_pesme=df,
    top_n=10
)

# Ispisujemo rezultate u identičnom formatu
if preporuke_menhetn:
    print(f"\nNOVI TOP 10 REZULTATI (Menhetn rastojanje - Muzička blizina):")
    print("-" * 75)
    for rank, p in enumerate(preporuke_menhetn, 1):
        # Ispisujemo udaljenost (distance) - što je manja, pesma je bolji pogodak!
        print(f"{rank}. [{p['song_id']}] (Udaljenost: {p['distance']:.4f})")
else:
    print("\nNema preporuka.")

In [ ]:
from sklearn.metrics.pairwise import manhattan_distances
import numpy as np
import pandas as pd

def generisi_manhattan_preporuke_safe(target_user_id, interakcije_df, konacni_vektori_pesama, song_id_u_indeks, df_pesme, top_n=10):
    # 1. Pronalazimo sve pesme koje je ovaj korisnik slušao
    korisnik_istorija = interakcije_df[interakcije_df['user_id'] == target_user_id]
    
    if korisnik_istorija.empty:
        return []

    # 2. Pravimo profil korisnika kao ponderisani prosek vektora pesama
    profil_korisnika = None
    ukupno_pustanja = 0
    
    for _, red in korisnik_istorija.iterrows():
        s_id = red['song_id']
        broj_slusanja = red['play_count']
        
        # Proveravamo da li pesma postoji u našoj matrici karakteristika
        if s_id in song_id_u_indeks:
            indeks_pesme = song_id_u_indeks[s_id]
            
            # Izvlačimo vektor te pesme iz CSR sparse matrice
            vektor_pesme = konacni_vektori_pesama.getrow(indeks_pesme).toarray()
            
            # Ponderišemo vektor brojem slušanja
            if profil_korisnika is None:
                profil_korisnika = vektor_pesme * broj_slusanja
            else:
                profil_korisnika += vektor_pesme * broj_slusanja
                
            ukupno_pustanja += broj_slusanja

    if profil_korisnika is None or ukupno_pustanja == 0:
        return []
        
    # Normalizacija profila korisnika
    profil_korisnika = profil_korisnika / ukupno_pustanja

    # 3. Računamo Menhetn rastojanje između profila i SVIH pesama u bazi
    # .toarray() se koristi jer manhattan_distances zahteva guste nizove
    udaljenosti = manhattan_distances(profil_korisnika, konacni_vektori_pesama.toarray())[0]

    # 4. Sortiramo pesme: manja udaljenost znači VEĆU sličnost!
    # argsort prirodno sortira od najmanjeg ka najvećem broju
    rangirani_indeksi = np.argsort(udaljenosti)

    # 5. Izvlačimo top N pesama, preskačući one koje je već slušao
    slusane_pesme_ids = set(korisnik_istorija['song_id'])
    preporuke = []
    
    for indeks in rangirani_indeksi:
        pesma_info = df_pesme.iloc[indeks]
        if pesma_info['song_id'] not in slusane_pesme_ids:
            preporuke.append({
                'song_id': pesma_info['song_id'],
                'distance': udaljenosti[indeks]  # Beležimo udaljenost (skor)
            })
        
        if len(preporuke) == top_n:
            break
            
    return preporuke


def evaluiraj_samo_manhattan(df_train, df_test, kombinovana_matrica, song_id_u_indeks, df_pesme, top_n=10):
    import scipy.sparse as sp
    from sklearn.metrics.pairwise import manhattan_distances
    
    # Sigurna konverzija u guste nizove
    if sp.issparse(kombinovana_matrica):
        M_gusta = kombinovana_matrica.toarray()
    elif hasattr(kombinovana_matrica, 'A'):
        M_gusta = kombinovana_matrica.A
    else:
        M_gusta = np.asarray(kombinovana_matrica)
        
    if M_gusta.ndim == 0:
        M_gusta = M_gusta.item()
        if sp.issparse(M_gusta):
            M_gusta = M_gusta.toarray()
            
    M_gusta = np.asarray(M_gusta, dtype=np.float64)
    
    ukupno_hitova = 0
    ukupno_preciznost = 0.0
    procenjeni_korisnici = 0
    
    test_users = df_test['user_id'].unique()
    df_pesme_reset = df_pesme.reset_index(drop=True)
    
    kolona_slusanja = 'play_count' if 'play_count' in df_train.columns else 'listen_count'
    
    for target_user_id in test_users:
        user_train = df_train[df_train['user_id'] == target_user_id]
        if user_train.empty:
            continue
            
        user_test_pesme = set(df_test[df_test['user_id'] == target_user_id]['song_id'])
        
        # Pravljenje profila
        profil_korisnika = None
        ukupno_pustanja = 0
        
        for _, red in user_train.iterrows():
            s_id = red['song_id']
            broj_slusanja = red[kolona_slusanja]
            if s_id in song_id_u_indeks:
                idx_p = song_id_u_indeks[s_id]
                if profil_korisnika is None:
                    profil_korisnika = M_gusta[idx_p] * broj_slusanja
                else:
                    profil_korisnika += M_gusta[idx_p] * broj_slusanja
                ukupno_pustanja += broj_slusanja
                
        if profil_korisnika is None or ukupno_pustanja == 0:
            continue
            
        profil_korisnika = (profil_korisnika / ukupno_pustanja).reshape(1, -1)
        
        # RAČUNAMO ISKLJUČIVO MENHETN
        manh_udaljenosti = manhattan_distances(profil_korisnika, M_gusta).flatten()
        
        # Sortiramo od NAJMANJE do NAJVEĆE udaljenosti
        manh_top_indeksi = np.argsort(manh_udaljenosti)
        
        skup_slusanih_id = set(user_train['song_id'])
        preporuke_id = []
        
        for idx in manh_top_indeksi:
            s_id = df_pesme_reset.at[idx, 'song_id']
            if s_id not in skup_slusanih_id:
                preporuke_id.append(s_id)
            if len(preporuke_id) == top_n:
                break
                
        pogoci = [pesma for pesma in preporuke_id if pesma in user_test_pesme]
        broj_pogodaka = len(pogoci)
        
        if broj_pogodaka > 0:
            ukupno_hitova += 1
        ukupno_preciznost += broj_pogodaka / top_n
        procenjeni_korisnici += 1

    prosecan_hr = ukupno_hitova / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    prosecna_preciznost = ukupno_preciznost / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    
    print("\n--- STVARNI REZULTATI ZA MENHETN (BEZ KEŠA) ---")
    print(f"Hit Rate@10:     {prosecan_hr:.4f}")
    print(f"Precision@10:    {prosecna_preciznost:.4f}")
    
    return prosecan_hr, prosecna_preciznost

# Pokretanje potpuno nove funkcije
hr_m, prec_m = evaluiraj_samo_manhattan(df_train, df_test, kombinovana_matrica, song_id_u_indeks, df, top_n=15)